[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ababber/pyhou-02-17-2026/blob/main/part-3-foundation-models/research-notebook.ipynb)

> **What You'll Need**
> - **To run demos locally:** Python 3.9+, torch, transformers, chronos-forecasting ([see requirements](../requirements.txt))
> - **To run demos in Colab:** Nothing — click the badge above, install chronos with `!pip install chronos-forecasting`
> - **To run the trading strategy:** Free [QuantConnect](https://www.quantconnect.com/) account (no credit card required)
> - **Time:** ~30 min to read, ~15 min to run backtest on QC (quarterly rebalancing = fast)



# Part 3: Foundation Models — Amazon Chronos

**Zero-Shot Time Series Forecasting**

In Part 1, ridge regression failed to generate alpha. In Part 2, a temporal CNN succeeded — but required 780 training runs. This notebook asks: what if you could skip the training entirely?

We'll use Amazon Chronos — a pre-trained transformer that has never seen financial data — and test whether it can forecast stock prices out of the box.

**What you'll learn:**
- How foundation models tokenize continuous time series into discrete tokens
- Why attention mechanisms work for temporal patterns
- How to use Chronos for zero-shot forecasting (no training)
- The trade-offs between base and fine-tuned models

**Prerequisites:**
- Parts 1 and 2 of this series (for backtest metric interpretation)
- Basic Python (numpy, pandas)
- Familiarity with transformer concepts (attention, encoder-decoder)

---

**Navigation:** [The Strategy](#2-the-strategy) | [The Math](#3-the-math) | [Implementation](#4-implementation) | [Results](#5-results) | [Analysis](#6-analysis) | [References](#7-references)

## 1. Introduction

### The Question

Can a pre-trained model — one that has *never* seen financial data — generate trading alpha?

This series tests three generations of ML on the same backtesting platform:

| Part | Model | Training Effort | Alpha | Beta | Verdict |
|------|-------|-----------------|-------|------|---------|
| 1 | Ridge Regression | Weekly per asset | -0.062 | 1.146 | Beta masquerade |
| 2 | Temporal CNN | 780 runs (5 years) | 0.093 | 0.278 | **Genuine alpha** |
| 3 | Chronos (Base) | **Zero** | 0.040 | 1.125 | Market + edge |
| 3 | Chronos (Fine-tuned) | **3 steps** | 0.076 | 1.110 | Better edge |

Same platform. Same time period. Radically different effort levels.

### Why Foundation Models?

**The paradigm shift:** Traditional ML requires training on financial data. Foundation models flip this — they're pre-trained on massive general datasets (weather, energy, retail, economics) and applied to finance without modification.

**Amazon Chronos** is built on T5 (Text-to-Text Transfer Transformer), the same architecture family as many large language models. But instead of processing words, it processes numbers by tokenizing them into discrete bins.

> **Key Insight:** Temporal patterns are universal. A "spike then mean reversion" looks the same whether it's temperature, sales, or stock prices. The model doesn't need to know it's looking at finance — it just recognizes shapes.

**Why this matters for trading:**
- **Zero training cost:** Load model, generate forecasts immediately
- **No overfitting risk:** Model hasn't seen your specific data
- **Transfer learning:** Patterns learned from billions of data points across domains

## 2. The Strategy

### Universe: Top 5 by Dollar Volume

Unlike Part 1's futures basket or Part 2's QQQ top-3, this strategy uses a simple liquidity filter:

| Selection | Method |
|-----------|--------|
| Universe | All US equities |
| Filter | Top 5 by dollar volume |
| Update | Monthly (first trading day) |

**Why dollar volume?** It's model-agnostic. We're not picking stocks based on sector, fundamentals, or technical signals. We're just trading the most liquid names — whatever they happen to be.

**Typical holdings:** AAPL, MSFT, NVDA, GOOGL, AMZN (the mega-caps that dominate daily trading volume).

### Features: None (Just Price History)

This is the radical departure from Parts 1 and 2:

| Part | Features | Engineering Effort |
|------|----------|--------------------|
| 1 | Volatility, ATR, Open Interest | Manual computation |
| 2 | Raw OHLCV (15×5 matrix) | Normalization + CNN learns |
| 3 | **Closing prices only** | **Nothing** |

Chronos takes raw closing prices as input. No feature engineering. No normalization (the model handles it internally). No lookback window design.

**Input:** 365 days of closing prices per stock
**Output:** Probabilistic forecast of next 63 trading days (one quarter)

### Allocation: Sharpe-Optimal Weights

Once Chronos generates forecasts, we optimize portfolio weights to maximize the forward Sharpe ratio:

```
max  (R_portfolio - R_f) / σ_portfolio
s.t. Σ w_i = 1       (fully invested)
     0 ≤ w_i ≤ 1     (no shorting)
```

**Why Sharpe optimization?** It balances return and risk. High-return forecasts get more weight, but only if they don't add too much portfolio volatility.

**Schedule:**
- **Forecast:** Quarterly (first trading day of Q1, Q2, Q3, Q4)
- **Optimize:** SciPy SLSQP on forecasted equity curves
- **Rebalance:** Full liquidation + re-entry based on optimal weights

> **Key Insight:** The model generates forecasts. The optimizer decides how to trade them. This separation lets us use the same Chronos model with different allocation strategies.

## 3. The Math

Three concepts drive Chronos: **tokenization** (converting numbers to tokens), **attention** (relating tokens to each other), and **probabilistic forecasting** (predicting distributions, not point estimates).

### Tokenization: Numbers → Tokens

LLMs process words by converting them to tokens (integer IDs). Chronos does the same for numbers:

**Step 1: Scale normalization**
$$x_{scaled} = \frac{x - \mu}{\sigma}$$

Where μ and σ are the mean and standard deviation of the input series. This makes the model scale-invariant — it doesn't matter if prices are $10 or $1000.

**Step 2: Uniform binning**
The scaled value is mapped to one of 4,096 discrete bins:

$$token = \lfloor \frac{x_{scaled} - x_{min}}{x_{max} - x_{min}} \times 4096 \rfloor$$

Now every price is a "word" the transformer can process.

### Demo: How Tokenization Works

Run this cell to see how Chronos converts prices to tokens.

> **Note:** This is a simplified demonstration. Chronos uses `MeanScaleUniformBins` which includes additional handling for edge cases and the special tokens. The core concept — scaling then binning — is the same.

In [1]:
# RUNNABLE DEMO: Chronos-style tokenization
import numpy as np

np.random.seed(42)

# Simulate 30 days of stock prices (starting at $150, ~1% daily moves)
prices = [150.0]
for _ in range(29):
    prices.append(prices[-1] * (1 + np.random.normal(0, 0.01)))
prices = np.array(prices)

# Step 1: Scale normalization (mean/std)
mu, sigma = prices.mean(), prices.std()
scaled = (prices - mu) / sigma

# Step 2: Uniform binning to 4096 tokens
n_bins = 4096
x_min, x_max = scaled.min(), scaled.max()
tokens = np.floor((scaled - x_min) / (x_max - x_min + 1e-8) * n_bins).astype(int)
tokens = np.clip(tokens, 0, n_bins - 1)

print("Chronos-Style Tokenization Demo")
print("=" * 50)
print(f"\nOriginal prices (first 5): {prices[:5].round(2)}")
print(f"Mean: ${mu:.2f}, Std: ${sigma:.2f}")
print(f"\nScaled values (first 5): {scaled[:5].round(3)}")
print(f"Token IDs (first 5): {tokens[:5]}")
print(f"\nToken range: {tokens.min()} to {tokens.max()} (of 0-4095)")
print(f"\n→ Each price is now a 'word' the transformer can process")

Chronos-Style Tokenization Demo

Original prices (first 5): [150.   150.75 150.54 151.51 153.82]
Mean: $149.84, Std: $4.68

Scaled values (first 5): [0.034 0.194 0.149 0.357 0.851]
Token IDs (first 5): [2209 2415 2358 2627 3265]

Token range: 0 to 4095 (of 0-4095)

→ Each price is now a 'word' the transformer can process


### Attention: Which History Matters?

The transformer's superpower is **attention** — the ability to dynamically weight which past tokens matter for predicting the future.

**Self-attention formula:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

| Term | Meaning |
|------|---------|
| Q (Query) | "What am I looking for?" |
| K (Key) | "What do I contain?" |
| V (Value) | "What do I contribute?" |
| √d_k | Scaling factor (prevents large dot products) |

**In Chronos:**
- **Encoder self-attention:** Each historical price token attends to all other historical tokens. "Given I'm at day 15, how relevant is day 3? Day 10?"
- **Decoder cross-attention:** Each predicted future token attends to the encoder's representation. "To predict day 31, which historical patterns matter?"

> **Key Insight:** Unlike CNNs (which use fixed-size kernels) or RNNs (which compress history into a single state), attention can directly connect any two time points. A pattern from 100 days ago can directly influence today's forecast.

### Demo: Simplified Attention Weights

This shows how attention dynamically weights history based on similarity.

> **Note:** Real transformer attention uses learned Query/Key/Value projections and multi-head attention. This demo uses raw price similarity to illustrate the core concept: attention dynamically decides which history matters for each prediction.

In [2]:
# RUNNABLE DEMO: Simplified self-attention
import numpy as np

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / exp_x.sum()

# Simulate a price pattern: uptrend → spike → mean reversion
pattern = np.array([100, 102, 104, 106, 108, 115, 110, 108, 107, 106])
days = [f"Day {i+1}" for i in range(len(pattern))]

# Query: "Which past days are similar to Day 10 (106)?"
query_idx = 9  # Day 10
query_value = pattern[query_idx]

# Simple similarity: negative absolute difference (closer = higher attention)
similarities = -np.abs(pattern - query_value)
attention_weights = softmax(similarities * 0.5)  # Temperature scaling

print("Self-Attention Demo: Which history matters for Day 10?")
print("=" * 55)
print(f"\nPrice pattern: {pattern.tolist()}")
print(f"Query (Day 10): ${query_value}")
print(f"\nAttention weights (which days get focus):")
for i, (day, price, weight) in enumerate(zip(days, pattern, attention_weights)):
    bar = "█" * int(weight * 50)
    marker = " ← QUERY" if i == query_idx else ""
    print(f"  {day} (${price}): {weight:.3f} {bar}{marker}")

print(f"\n→ Days with similar prices (${query_value}) get highest attention")
print(f"→ The spike day (Day 6, ${pattern[5]}) gets almost zero attention")

Self-Attention Demo: Which history matters for Day 10?

Price pattern: [100, 102, 104, 106, 108, 115, 110, 108, 107, 106]
Query (Day 10): $106

Attention weights (which days get focus):
  Day 1 ($100): 0.012 
  Day 2 ($102): 0.033 █
  Day 3 ($104): 0.091 ████
  Day 4 ($106): 0.247 ████████████
  Day 5 ($108): 0.091 ████
  Day 6 ($115): 0.003 
  Day 7 ($110): 0.033 █
  Day 8 ($108): 0.091 ████
  Day 9 ($107): 0.150 ███████
  Day 10 ($106): 0.247 ████████████ ← QUERY

→ Days with similar prices ($106) get highest attention
→ The spike day (Day 6, $115) gets almost zero attention


### Probabilistic Forecasting

Chronos doesn't output a single predicted price. It outputs a **distribution** over all 4,096 possible tokens for each future timestep.

**Output format:** For each future day t, the model produces:
- Probability distribution P(token | history) over 4,096 bins
- We extract the median (0.5 quantile) as the point forecast
- Uncertainty bands can be computed from other quantiles (0.1, 0.9)

**Why distributions matter:**
1. **Uncertainty quantification:** We know when the model is confident vs. uncertain
2. **Risk management:** Wide prediction intervals → smaller position sizes
3. **Ensemble effect:** The distribution implicitly averages over many possible futures

## 4. Implementation

### Two Strategies: Base vs. Fine-Tuned

We test two variants of the same architecture:

| Strategy | Training | Sharpe | Alpha | Drawdown |
|----------|----------|--------|-------|----------|
| **Base Model** | Zero (pre-trained weights only) | 0.727 | 0.040 | 41.2% |
| **Fine-Tuned** | 3 gradient steps per quarter | 0.846 | 0.076 | 49.5% |

**Base Model:** Load `amazon/chronos-t5-tiny` from HuggingFace. Generate forecasts. Trade.

**Fine-Tuned:** Same, but at each quarterly rebalance, first fine-tune on the past year of equity data (3 gradient steps, learning_rate=1e-5). Then forecast and trade.

### Demo: Using Chronos for Forecasting

This shows the core forecasting logic (requires `chronos-forecasting` package).

> **Note:** If `chronos-forecasting` isn't installed, the cell prints a description of what the code does. In Colab, run `!pip install chronos-forecasting` first.

In [3]:
# RUNNABLE DEMO: Chronos forecasting (requires chronos-forecasting package)
# Install: pip install chronos-forecasting

try:
    import torch
    from chronos import ChronosPipeline
    import numpy as np

    # Load the pre-trained model
    pipeline = ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-tiny",
        device_map="cpu",
        torch_dtype=torch.float32,
    )

    # Generate sample price history (365 days)
    np.random.seed(42)
    prices = [150.0]
    for _ in range(364):
        prices.append(prices[-1] * (1 + np.random.normal(0.0003, 0.015)))
    context = torch.tensor(prices)

    # Forecast next 63 days (one quarter)
    forecast = pipeline.predict(
        context,
        prediction_length=63,
        num_samples=20,  # Generate 20 sample paths
    )

    # Extract median forecast
    median_forecast = np.median(forecast[0].numpy(), axis=0)

    print("Chronos Forecasting Demo")
    print("=" * 50)
    print(f"Input: {len(prices)} days of price history")
    print(f"Last price: ${prices[-1]:.2f}")
    print(f"\nForecast (next 5 days):")
    for i in range(5):
        print(f"  Day {i+1}: ${median_forecast[i]:.2f}")
    print(f"\nForecast range: ${median_forecast.min():.2f} to ${median_forecast.max():.2f}")
    print(f"→ Model generated {len(median_forecast)} day forecast with zero training")

except ImportError:
    print("chronos-forecasting not installed")
    print("Install with: pip install chronos-forecasting")
    print("\nThis demo shows how Chronos generates forecasts:")
    print("  1. Load pre-trained model from HuggingFace")
    print("  2. Pass price history as context tensor")
    print("  3. Get probabilistic forecast (multiple sample paths)")
    print("  4. Extract median for point estimate")

chronos-forecasting not installed
Install with: pip install chronos-forecasting

This demo shows how Chronos generates forecasts:
  1. Load pre-trained model from HuggingFace
  2. Pass price history as context tensor
  3. Get probabilistic forecast (multiple sample paths)
  4. Extract median for point estimate


### Portfolio Optimization

Once we have forecasts for each stock, we optimize weights to maximize the Sharpe ratio:

$$\text{Sharpe} = \frac{\mathbb{E}[R_p] - R_f}{\sigma_p} = \frac{\sum_i w_i \bar{r}_i \cdot 252 - R_f}{\sqrt{w^T \Sigma w \cdot 252}}$$

The optimization uses SciPy's SLSQP (Sequential Least Squares Programming) with:
- **Equality constraint:** Weights sum to 1 (fully invested)
- **Bound constraints:** Each weight between 0 and 1 (no shorting)

In [4]:
# RUNNABLE DEMO: Sharpe ratio portfolio optimization
import numpy as np
from scipy.optimize import minimize

np.random.seed(42)

# Simulated forecasted returns for 5 stocks (63 days)
n_stocks, n_days = 5, 63
stock_names = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN"]

# Generate realistic-ish forecasted daily returns
forecasted_returns = np.random.normal(0.0005, 0.02, (n_stocks, n_days))
forecasted_returns[2] += 0.001  # NVDA has higher expected return

def neg_sharpe(weights, returns, risk_free_rate=0.05):
    """Negative Sharpe ratio (for minimization)."""
    portfolio_return = np.sum(weights * returns.mean(axis=1)) * 252
    portfolio_vol = np.sqrt(np.dot(weights.T, np.dot(np.cov(returns) * 252, weights)))
    return -(portfolio_return - risk_free_rate) / portfolio_vol

# Optimization
n = len(stock_names)
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}  # Weights sum to 1
bounds = [(0, 1) for _ in range(n)]  # No shorting
initial_weights = np.ones(n) / n  # Start with equal weights

result = minimize(
    neg_sharpe, initial_weights,
    args=(forecasted_returns,),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

optimal_weights = result.x

print("Portfolio Optimization Demo")
print("=" * 50)
print("\nForecasted annualized returns per stock:")
for name, ret in zip(stock_names, forecasted_returns.mean(axis=1) * 252):
    print(f"  {name}: {ret:+.1%}")

print("\nOptimal weights (Sharpe-maximizing):")
for name, weight in zip(stock_names, optimal_weights):
    bar = "█" * int(weight * 30)
    print(f"  {name}: {weight:.1%} {bar}")

print(f"\nOptimized Sharpe ratio: {-result.fun:.3f}")
print(f"→ NVDA gets highest weight due to highest forecasted return")

Portfolio Optimization Demo

Forecasted annualized returns per stock:
  AAPL: -75.8%
  MSFT: +46.3%
  NVDA: +54.4%
  GOOGL: +43.1%
  AMZN: +46.2%

Optimal weights (Sharpe-maximizing):
  AAPL: 0.0% 
  MSFT: 25.1% ███████
  NVDA: 32.7% █████████
  GOOGL: 15.0% ████
  AMZN: 27.3% ████████

Optimized Sharpe ratio: 2.817
→ NVDA gets highest weight due to highest forecasted return


### End-to-End Pipeline: Chronos → Optimizer

This cell connects forecasting to portfolio optimization in 5 steps:

1. **Data prep** — 365 days of historical prices for 5 tickers
2. **Chronos forward pass** — Generate 63-day probabilistic forecasts
3. **Tensor → NumPy handoff** — Extract median from `[1, num_samples, 63]` tensor
4. **Returns transform** — Convert prices to daily returns, annualize
5. **SLSQP optimization** — Maximize Sharpe with long-only constraints

The code is heavily commented — follow along to see exactly where PyTorch ends and NumPy begins.

> **Note:** This cell requires a GPU runtime in Colab. Go to Runtime → Change runtime type → T4 GPU.

In [6]:
!pip install chronos-forecasting

import torch
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from chronos import ChronosPipeline

# --- 1. Setup and Data Preparation ---
np.random.seed(42)
tickers = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'META']
num_days = 365

# Simulate daily prices using geometric random walk
prices = {}
for ticker in tickers:
    initial_price = np.random.uniform(100, 300)
    returns = np.random.normal(0.0005, 0.015, num_days)
    prices[ticker] = initial_price * np.cumprod(1 + returns)

df_prices = pd.DataFrame(prices)

# --- 2. Chronos Forecasting ---
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny",
    device_map="cpu",
    dtype=torch.float32,
)

prediction_length = 63
num_samples = 20
forecasts_dict = {}

print("Generating forecasts...")
for ticker in tickers:
    context = torch.tensor(df_prices[ticker].values, dtype=torch.float32)
    forecast = pipeline.predict(
        context,
        prediction_length=prediction_length,
        num_samples=num_samples,
    )
    # Extract median forecast and convert to 1D numpy array
    median_forecast = np.median(forecast[0].numpy(), axis=0)
    forecasts_dict[ticker] = median_forecast

df_forecasts = pd.DataFrame(forecasts_dict)

# --- 3. Returns Transformation ---
# Calculate expected daily returns from the forecasted prices
# pct_change() will result in NaN for the first row, so we drop it
df_returns = df_forecasts.pct_change().dropna()

# Calculate expected return vector and covariance matrix (annualized)
expected_returns = df_returns.mean() * 252
cov_matrix = df_returns.cov() * 252

# --- 4. Portfolio Optimization ---
def neg_sharpe_ratio(weights, expected_returns, cov_matrix):
    # Assuming 0% risk-free rate for simplicity
    portfolio_return = np.dot(weights, expected_returns)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return -portfolio_return / portfolio_volatility

num_assets = len(tickers)
initial_weights = np.ones(num_assets) / num_assets

# Constraints and bounds
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}) # Weights sum to 1
bounds = tuple((0.0, 1.0) for _ in range(num_assets)) # Long-only

print("Optimizing portfolio weights...")
result = minimize(
    neg_sharpe_ratio,
    initial_weights,
    args=(expected_returns.values, cov_matrix.values),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

# --- 5. Output ---
print("\nOptimal Portfolio Weights:")
optimal_weights = pd.Series(result.x, index=tickers)
print(optimal_weights.round(4))
print(f"\nExpected Annualized Portfolio Return: {np.dot(result.x, expected_returns):.4f}")
print(f"Expected Annualized Portfolio Volatility: {np.sqrt(np.dot(result.x.T, np.dot(cov_matrix, result.x))):.4f}")
print(f"Optimized Sharpe Ratio: {-result.fun:.4f}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Generating forecasts...
Optimizing portfolio weights...

Optimal Portfolio Weights:
AAPL    0.4860
MSFT    0.2758
GOOG    0.0512
AMZN    0.1322
META    0.0547
dtype: float64

Expected Annualized Portfolio Return: 0.2025
Expected Annualized Portfolio Volatility: 0.0566
Optimized Sharpe Ratio: 3.5754


## 5. Results

### Backtest Performance (2019-2024)

| Metric | Base Model | Fine-Tuned | Better |
|--------|------------|------------|--------|
| Net Profit | +199.9% | +266.4% | FT |
| CAGR | 23.2% | 28.0% | FT |
| Sharpe | 0.727 | 0.846 | FT |
| Sortino | 0.804 | 0.923 | FT |
| Max Drawdown | 41.2% | 49.5% | Base |
| Alpha | 0.040 | 0.076 | FT |
| Beta | 1.125 | 1.110 | FT |
| PSR | 25.7% | 34.6% | FT |
| Total Orders | 104 | 96 | FT |
| Capacity | $1.4B | $920M | Base |

**Key observations:**
1. Fine-tuning improves Sharpe from 0.727 → 0.846 (+16%)
2. Alpha nearly doubles: 0.04 → 0.076
3. But drawdown increases: 41% → 50%
4. Both strategies have beta >1.1 (leveraged market exposure)

### Cross-Series Comparison

| Strategy | Sharpe | Net Profit | Alpha | Beta | Training Effort |
|----------|--------|------------|-------|------|-----------------|
| Ridge (Part 1) | 0.212 | +35% | -0.062 | 1.146 | Weekly/asset |
| CNN (Part 2) | 0.649 | +145% | 0.093 | 0.278 | 780 runs |
| Chronos Base | 0.727 | +200% | 0.040 | 1.125 | **Zero** |
| Chronos FT | 0.846 | +266% | 0.076 | 1.110 | 3 steps/quarter |

**The pattern:** Less effort, better results.

**The nuance:** CNN has the cleanest alpha (0.093 with beta 0.278). Chronos has higher total returns but most comes from market exposure (beta >1.1).

## 6. Analysis

### Caveats and Limitations

**1. Lookahead Bias**
Chronos was released in March 2024. It was trained on data that likely includes patterns from our 2019-2024 backtest period. The base model's performance may be inflated.

**2. Statistical Significance**
No strategy achieves 95% PSR (Probabilistic Sharpe Ratio):

| Strategy | PSR | Interpretation |
|----------|-----|----------------|
| Ridge | 2.4% | Almost certainly noise |
| CNN | 21.9% | Suggestive, not definitive |
| Chronos Base | 25.7% | Suggestive, not definitive |
| Chronos FT | 34.6% | Better, still not significant |

**3. Different Universes**
We're comparing different asset classes:
- Ridge: Futures (12 contracts across indices/energy/grains)
- CNN: QQQ top-3 (mega-cap tech)
- Chronos: Top-5 by dollar volume (mega-cap mixed)

**4. Survivorship Bias**
Universe uses current ETF constituents, not historical.

**5. Small Universes**
3-5 stocks at a time limits diversification benefits.

**6. Tokenization Normalization**
When Chronos scales prices to tokens, it uses mean (μ) and standard deviation (σ). If these are calculated over the entire window (including future data), information leaks into historical tokens. Production implementations must use a rolling or expanding window of strictly historical data — normalize price at time *t* using only data from *t-k* to *t-1*.

**7. Trading Frictions**
The reported alpha (0.04–0.076) is gross, not net. Full quarterly liquidation and reallocation incurs:
- **Slippage:** At $1.4B capacity, moving hundreds of millions moves prices against you
- **Transaction costs:** SEC fees, exchange fees, bid-ask spreads compound over quarterly rebalances
- **No turnover constraint:** The optimizer may suggest drastic weight changes (0% → 100%) to chase marginal Sharpe improvements, incurring massive turnover costs

A gross alpha of 4% could be entirely consumed by these frictions, leaving net alpha at zero or negative.

### Demo: Probabilistic Sharpe Ratio (PSR)

PSR answers: "What's the probability this Sharpe ratio is genuinely positive?"

In [7]:
# RUNNABLE DEMO: Probabilistic Sharpe Ratio
import numpy as np
from scipy import stats

def probabilistic_sharpe_ratio(sharpe, n_samples, skewness=0, kurtosis=3):
    """
    Calculate the probability that true Sharpe > 0 given observed Sharpe.

    Based on Bailey & López de Prado (2012).
    """
    se = np.sqrt((1 + 0.5 * sharpe**2 - skewness * sharpe +
                  (kurtosis - 3) / 4 * sharpe**2) / n_samples)
    z_score = sharpe / se
    return stats.norm.cdf(z_score)

# Actual PSR values from QuantConnect backtests (computed from real return distributions)
# These account for actual skewness and kurtosis in the return series
qc_psr = {
    "Ridge (Part 1)": 0.0244,    # 2.44%
    "CNN (Part 2)": 0.2192,      # 21.92%
    "Chronos Base": 0.2574,      # 25.74%
    "Chronos FT": 0.3464,        # 34.64%
}

print("Probabilistic Sharpe Ratio — Authoritative Values from QuantConnect")
print("=" * 65)
print(f"{'Strategy':<20} {'Sharpe':>10} {'PSR':>12} {'Significant?':>15}")
print("-" * 65)

sharpes = {"Ridge (Part 1)": 0.212, "CNN (Part 2)": 0.649,
           "Chronos Base": 0.727, "Chronos FT": 0.846}

for name, psr in qc_psr.items():
    sharpe = sharpes[name]
    sig = "✓ Yes" if psr > 0.95 else "✗ No"
    print(f"{name:<20} {sharpe:>10.3f} {psr:>11.1%} {sig:>15}")

print("-" * 65)
print("\nNote: PSR accounts for skewness and kurtosis in actual return distributions.")
print("95% threshold = statistically significant at 5% level.")
print("→ None of our strategies pass. Results are suggestive, not definitive.")

Probabilistic Sharpe Ratio — Authoritative Values from QuantConnect
Strategy                 Sharpe          PSR    Significant?
-----------------------------------------------------------------
Ridge (Part 1)            0.212        2.4%            ✗ No
CNN (Part 2)              0.649       21.9%            ✗ No
Chronos Base              0.727       25.7%            ✗ No
Chronos FT                0.846       34.6%            ✗ No
-----------------------------------------------------------------

Note: PSR accounts for skewness and kurtosis in actual return distributions.
95% threshold = statistically significant at 5% level.
→ None of our strategies pass. Results are suggestive, not definitive.


### Alpha vs. Returns: A Nuance

**Alpha ≠ Returns.** This is crucial for understanding the results.

| Strategy | Alpha | Beta | What it means |
|----------|-------|------|---------------|
| CNN | 0.093 | 0.278 | Cleanest alpha — mostly independent returns |
| Chronos FT | 0.076 | 1.110 | Higher returns, but mostly market exposure |

**CNN:** Low beta means returns are largely independent of the market. On average, when the market moves, the CNN strategy moves only ~28% as much. The alpha is relatively "pure" signal.

**Chronos:** High beta means the strategy amplifies market moves on average. But note: beta is a *statistical expectation*, not an exact multiplier. Individual days will vary.

**Important caveats on this comparison:**

1. **The benchmark matters.** Both alpha and beta are calculated relative to a benchmark (typically SPY). Chronos trades top-5 mega-caps by dollar volume (AAPL, MSFT, NVDA, etc.) — stocks that *inherently* have high beta relative to SPY because they dominate the index. The 1.110 beta partially reflects the universe, not just the model's allocations.

2. **Different universes, limited comparability.** CNN trades QQQ top-3; Chronos trades top-5 by volume. Comparing alpha across strategies that trade fundamentally different universes is methodologically weak. Some of CNN's "purer" alpha may simply reflect asset selection during a specific timeframe.

**Which is better?** Depends on your goal:
- **Market-neutral hedge fund:** CNN's lower beta is more valuable
- **Long-biased portfolio:** Chronos gives higher total returns

But neither comparison is apples-to-apples.

## 7. References

### Primary Sources

1. **Ansari, A. F. et al. (2024).** "Chronos: Learning the Language of Time Series." *arXiv:2403.07815*. [[paper](https://arxiv.org/abs/2403.07815)]

2. **Pik, J., Chan, E. P., Broad, J., Sun, P., & Singh, V. (2025).** *Hands-On AI Trading with Python, QuantConnect, and AWS.* Wiley. ISBN 9781394268436. Exercise 18.

3. **Raffel, C. et al. (2020).** "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer." *JMLR 21(140)*. [[paper](https://arxiv.org/abs/1910.10683)]

### Statistical Foundations

4. **Sharpe, W. F. (1994).** "The Sharpe Ratio." *The Journal of Portfolio Management*, 21(1), 49-58.

5. **Bailey, D. H., & López de Prado, M. (2012).** "The Sharpe Ratio Efficient Frontier." *Journal of Risk*, 15(2).

6. **Markowitz, H. (1952).** "Portfolio Selection." *The Journal of Finance*, 7(1), 77-91.

### Tools

- **QuantConnect:** [quantconnect.com](https://www.quantconnect.com/)
- **Chronos on HuggingFace:** [huggingface.co/amazon/chronos-t5-tiny](https://huggingface.co/amazon/chronos-t5-tiny)
- **Code for this series:** [github.com/ababber/pyhou-02-17-2026](https://github.com/ababber/pyhou-02-17-2026)

## Summary

### What We Learned

**1. Foundation models work for finance — with zero training.**
Chronos achieves Sharpe 0.727 out of the box, beating ridge regression (0.212) without any financial data in its training set.

**2. Fine-tuning helps, but has costs.**
Three gradient steps improve Sharpe from 0.727 → 0.846, but drawdown increases from 41% → 50%. More isn't always better.

**3. Less effort, better results.**
| Model | Training Effort | Sharpe |
|-------|-----------------|--------|
| Ridge | Weekly/asset | 0.212 |
| CNN | 780 runs | 0.649 |
| Chronos | 0–3 steps | 0.727–0.846 |

**4. Alpha and returns aren't the same.**
CNN generates the purest alpha (low beta). Chronos generates higher total returns (high beta). Choose based on your portfolio goals.

**5. Nothing here is statistically significant.**
PSR < 35% for all strategies. These results are *interesting*, not *proven*.

### The Foundation Model Era

The tools are available:
- HuggingFace has Chronos ready to download
- QuantConnect has the backtesting infrastructure
- The barrier to entry has never been lower

What will you build?

---

*Educational content only. Not financial advice.*